# 📄 Resume Screening AI
### WeIntern · Artificial Intelligence · Week 2 · Task 2

**Goal:** read resumes, extract skills, education and experience, and rank candidates for a job description.

| Feature from the brief | Where |
|---|---|
| Accept resumes in PDF or text format | Steps 3 and 5 |
| Clean the text (NLP preprocessing) | Step 4 |
| Extract skills, education and experience | Step 6 |
| Accept a job description | Step 2 |
| Similarity score (TF-IDF + cosine similarity) | Step 7 |
| Rank and display candidates | Steps 8 and 9 |
| Summary report per candidate | Step 10 |

**Data:** click **+ Add Input** and add the *Resume Dataset* by Gaurav Dutta (`UpdatedResumeDataSet.csv`). Without a dataset, 4 sample resumes are used.

In [ ]:
# Step 1: Import libraries
import os, re, glob
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Step 2: The job description we want to hire for
job_description = """
Data Scientist
Requirements: 2+ years of experience, Bachelor's degree.
Skills: Python, SQL, Machine Learning, Deep Learning, Pandas, NumPy,
Scikit-learn, TensorFlow, NLP, Statistics, Data Visualization, Tableau
"""

In [ ]:
# Step 3: Load resumes
# Uses the Kaggle resume dataset if it is attached, otherwise 4 sample resumes.
csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

if csv_files:
    df = pd.read_csv(csv_files[0])
    text_col = "Resume" if "Resume" in df.columns else "Resume_str"
    df = df.rename(columns={text_col: "resume"}).drop_duplicates(subset="resume").reset_index(drop=True)
    df["name"] = ["Candidate " + str(i + 1) for i in range(len(df))]
    print("Loaded", len(df), "resumes from", csv_files[0])
else:
    df = pd.DataFrame({
        "name": ["Priya Sharma", "Rahul Verma", "Ananya Iyer", "Karan Mehta"],
        "resume": [
            "Data Scientist with 4 years of experience. B.Tech Computer Science. Python, SQL, Machine Learning, "
            "Deep Learning, TensorFlow, Pandas, NumPy, Scikit-learn, NLP, Statistics, Tableau, Data Visualization.",
            "ML Engineer with 3 years of experience. M.Tech AI. Python, PyTorch, Deep Learning, Computer Vision, "
            "NumPy, Docker, Kubernetes, SQL.",
            "Data Analyst with 3 years of experience. B.Sc Statistics. SQL, Excel, Power BI, Tableau, Python, "
            "Pandas, Statistics, Data Visualization.",
            "Frontend Developer with 2 years of experience. BCA. JavaScript, React, HTML, CSS, Node.js, MongoDB, Git.",
        ],
    })
    print("No dataset attached - using", len(df), "sample resumes")
df.head()

In [ ]:
# Step 4: Clean the text (lowercase, remove symbols and extra spaces)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|\S+@\S+", " ", text)      # remove links and emails
    text = re.sub(r"[^a-z0-9+#.\- ]", " ", text)      # keep letters, numbers, + # . -
    return re.sub(r"\s+", " ", text).strip()

df["clean"] = df["resume"].apply(clean_text)
print(df["clean"][0][:200])

In [ ]:
# Step 5: Read PDF or TXT resume files
def read_resume_file(path):
    if path.lower().endswith(".pdf"):
        try:
            from pypdf import PdfReader
        except ImportError:
            from PyPDF2 import PdfReader
        return " ".join(page.extract_text() or "" for page in PdfReader(path).pages)
    with open(path, encoding="utf-8", errors="ignore") as f:
        return f.read()

# Demo: read a few PDF resumes if the attached dataset has them
pdf_files = glob.glob("/kaggle/input/**/*.pdf", recursive=True)[:3]
for path in pdf_files:
    print(os.path.basename(path), "->", clean_text(read_resume_file(path))[:120], "...")
if not pdf_files:
    print("No PDF files attached - the resumes come from the CSV / samples above.")

In [ ]:
# Step 6: Find skills, education and experience in each resume
SKILLS = ["python", "java", "javascript", "sql", "r programming", "machine learning", "deep learning",
          "nlp", "computer vision", "tensorflow", "pytorch", "keras", "scikit-learn", "pandas", "numpy",
          "statistics", "data visualization", "tableau", "power bi", "excel", "spark", "hadoop",
          "aws", "azure", "docker", "kubernetes", "git", "linux", "django", "flask", "react", "node.js",
          "html", "css", "mongodb", "mysql", "communication", "leadership"]

def find_skills(text):
    return [s for s in SKILLS if re.search(r"(?<![a-z])" + re.escape(s) + r"(?![a-z])", text)]

def find_education(text):
    if re.search(r"ph\.?d", text): return "PhD"
    if re.search(r"m\.?tech|m\.?sc|mca|mba|master", text): return "Master's"
    if re.search(r"b\.?tech|b\.e\b|b\.?sc|bca|b\.?com|bachelor", text): return "Bachelor's"
    return "Not found"

def find_experience(text):
    years = [int(y) for y in re.findall(r"(\d+)\+?\s*years? of experience", text)]
    months = [int(m) / 12 for m in re.findall(r"(?:experience|exprience)\s*-?\s*(\d+)\s*months", text)]
    return round(max(years + months), 1) if years + months else 0

df["skills"] = df["clean"].apply(find_skills)
df["education"] = df["clean"].apply(find_education)
df["experience"] = df["clean"].apply(find_experience)
df[["name", "skills", "education", "experience"]].head()

### How the score works
- **Skill match (50%)**: share of the job's skills found in the resume
- **Text similarity (30%)**: TF-IDF cosine similarity between the job description and the resume
- **Experience (20%)**: candidate's years compared with the years required

In [ ]:
# Step 7: Score every resume against the job
job_clean = clean_text(job_description)
job_skills = find_skills(job_clean)
required_years = 2

# Text similarity with TF-IDF
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
vectors = tfidf.fit_transform([job_clean] + df["clean"].tolist())
df["text_similarity"] = cosine_similarity(vectors[0], vectors[1:]).flatten()

# Skill match and experience match
df["matched_skills"] = df["skills"].apply(lambda s: [x for x in job_skills if x in s])
df["missing_skills"] = df["skills"].apply(lambda s: [x for x in job_skills if x not in s])
df["skill_match"] = df["matched_skills"].apply(len) / len(job_skills)
df["experience_match"] = (df["experience"] / required_years).clip(upper=1)

# Final score out of 100
df["score"] = (100 * (0.5 * df["skill_match"] + 0.3 * df["text_similarity"]
                      + 0.2 * df["experience_match"])).round(1)
print("Job skills:", job_skills)

In [ ]:
# Step 8: Rank candidates (highest score first)
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)
ranked.index = ranked.index + 1          # rank starts at 1
ranked[["name", "score", "skill_match", "text_similarity", "experience", "education"]].head(10).round(2)

In [ ]:
# Step 9: Chart of the top 10 candidates
top = ranked.head(10).iloc[::-1]
plt.figure(figsize=(9, 5))
plt.barh(top["name"], top["score"], color="#2a78d6")
for i, s in enumerate(top["score"]):
    plt.text(s + 1, i, str(s), va="center")
plt.xlim(0, 100)
plt.xlabel("Score (out of 100)")
plt.title("Top candidates for: Data Scientist")
plt.tight_layout()
plt.show()

In [ ]:
# Step 10: Summary report for the top 3 candidates
for rank, row in ranked.head(3).iterrows():
    print(f"#{rank}  {row['name']}  -  Score: {row['score']}/100")
    print("   Education :", row["education"], "| Experience:", row["experience"], "years")
    print("   Matched   :", ", ".join(row["matched_skills"]))
    print("   Missing   :", ", ".join(row["missing_skills"]) or "none")
    print()

In [ ]:
# Step 11: Accuracy check (only if the dataset has job categories)
if "Category" in ranked.columns:
    top10 = ranked.head(10)
    correct = top10["Category"].str.contains("Data Science", case=False).sum()
    print(f"{correct} of the top 10 resumes are actually 'Data Science' resumes")
    print(top10[["name", "Category", "score"]])
else:
    print("No categories in the data - skipping accuracy check")

In [ ]:
# Step 12: Save the ranking as a CSV file (find it in the Output tab)
ranked[["name", "score", "education", "experience", "matched_skills", "missing_skills"]].to_csv("ranking.csv")
print("Saved ranking.csv")

### Conclusion
- Resumes are cleaned, skills, education and experience are extracted, and every candidate gets a score out of 100.
- Candidates are ranked, and the report shows which job skills each one has and which are missing.
- **Next steps:** add more skills, use Sentence-BERT for better text matching, and support scanned PDFs with OCR.